# Lesson 07 — 从 ADC 数字读数得到可信电压

这是一段真实的 C++ 实验：你修改代码格，再运行，就会看到下方深色虚拟设备的输入、输出和检查结果变化。Python 只负责让 VS Code Notebook 启动 C++ 编译和实验台。

## 先运行

按顺序运行初始化格和后面的代码格。绿色 ✓ 表示本格 C++ 符合当前契约；红色 ✗ 会给出预期与实际输出。


In [ ]:
# 运行支架：只运行一次，不需要学习或修改 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


ADC 给你的只是数字。要把旋钮或传感器读数用于决策，必须知道量化范围、参考电压和故障边界。

## 最小知识

本实验用 12 位 ADC：0 对应 0 V，4095 对应 3.3 V。`reading` 输出电压，2.0 V 以上亮 LED；故障时读数不可当成真实信号。


## 1. 第一个实验（Guided）

先运行。不要急着读每一行；先操作实验台，观察输入与输出之间的关系。


In [ ]:
%%academy_lab 07-adc-sensor guided guided pass
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            b_.output.reading = b_.input.analog * 3.3 / 4095.;
            b_.output.led = b_.output.reading >= 2.;
        }

    private:
        Device& b_;
    };
}


## 2. 修改一次

将触发阈值从 2.0 改为 1.0 V，拖动输入并观察量化阶梯。

重新运行这一个格，再在实验台确认你的预测。


In [ ]:
%%academy_lab 07-adc-sensor modify
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            b_.output.reading = b_.input.analog * 3.3 / 4095.;
            b_.output.led = b_.output.reading >= 2.;
        }

    private:
        Device& b_;
    };
}


## 3. 故障实验（Debug）

下面公式除以 4096 且忽略 fault；端点和故障输入会被悄悄误判。运行后失败是预期现象。用面板的时间线、波形、事务或终端找出失败发生在何时。


In [ ]:
%%academy_lab 07-adc-sensor broken exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            b_.output.reading = b_.input.analog * 3.3 / 4095.;
            b_.output.led = b_.output.reading >= 2.;
        }

    private:
        Device& b_;
    };
}


## 4. 你来完成（Build）

实现 0–3.3 V 换算、阈值输出和 fault 安全状态。

默认这格会失败，失败信息是需求的一部分。先自己尝试；需要时再看 [分层 Hint](../lessons/07-adc-sensor/hints.md)。


In [ ]:
%%academy_lab 07-adc-sensor exercise exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() { // TODO: ADC 到电压、阈值、fault
        }

    private:
        Device& b_;
    };
}


## 5. 开放挑战

加入多次采样平均，并解释它能降低什么噪声、不能修复什么接线问题。

## 你刚刚真正学到了什么？

ADC 读数必须带单位、量程和可信度。数值计算与传感器诊断是两件事。

## 硬件迁移

STEP 2 要核对 ESP32 ADC 衰减、校准、非线性和实际参考条件；本机面板不证明这些电气特性。

完成自己的尝试后才查看 [独立 Solution Notebook](solutions/07-adc-sensor-solution.ipynb)。
